In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import argparse

def load_attention_data(exp_dir, epoch):
    attn_path = exp_dir / f"attention_epoch_{epoch:04d}" / "attentions.pt"
    images_path = exp_dir / f"attention_epoch_{epoch:04d}" / "images.pt"
    attentions = torch.load(attn_path)
    images = torch.load(images_path)
    return attentions, images

def visualize_single(attentions, images, img_idx, layer_idx, head_idx, patch_size=4, save_path=None):
    img = images[img_idx].permute(1,2,0).numpy()
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)

    attn_layer = attentions[layer_idx][img_idx]
    attn_cls = attn_layer[head_idx, 0, 1:].cpu().numpy()
    num_patches_side = int(np.sqrt(attn_cls.shape[0]))
    attn_map = attn_cls.reshape(num_patches_side, num_patches_side)
    attn_map_resized = np.kron(attn_map, np.ones((patch_size, patch_size)))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10,4))
    ax1.imshow(img)
    ax1.set_title(f"Image {img_idx}")
    ax1.axis('off')

    ax2.imshow(img, alpha=0.3)
    im = ax2.imshow(attn_map_resized, cmap='jet', alpha=0.7)
    ax2.set_title(f"Layer {layer_idx}, Head {head_idx}")
    ax2.axis('off')
    plt.colorbar(im, ax=ax2, fraction=0.046, pad=0.04)

    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=100)
        plt.close()
    else:
        plt.show()

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--exp_dir", type=str, required=True)
    parser.add_argument("--epoch", type=int, required=True)
    parser.add_argument("--img_idx", type=int, default=0)
    parser.add_argument("--layer", type=int, default=0)
    parser.add_argument("--head", type=int, default=0)
    parser.add_argument("--patch_size", type=int, default=4)
    parser.add_argument("--save", type=str, default=None)
    args = parser.parse_args()

    exp_dir = Path(args.exp_dir)
    attentions, images = load_attention_data(exp_dir, args.epoch)
    visualize_single(attentions, images, args.img_idx, args.layer, args.head,
                     patch_size=args.patch_size, save_path=args.save)